In [81]:
import pandas as pd

from sklearn.metrics import cohen_kappa_score
from sentence_transformers import SentenceTransformer, util
import torch

In [82]:
path_1 = "annotations/2-TVSUM-video_5-6319373.txt"
path_2 = "annotations/2-TVSUM-video_5-6328583.txt"

In [83]:
def parse_file(filepath):
    rows = []
    
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            #Remove whitespace or empty lines
            line = line.strip()
            if not line:
                continue
            
            #Split the different parts with the comma
            parts = line.split(", ")
            description = parts[0]
            timestamp = parts[1]
            score = int(parts[2])
            
            rows.append({
                "description": description,
                "timestamp": timestamp,
                "score": score
            })
            
    return pd.DataFrame(rows)


In [84]:
annotator_1 = parse_file(path_1)
annotator_2 = parse_file(path_2)

annotator_2.head()

,description,timestamp,score
0,"Advertising title screen for the company ""GMC""...",00:00 - 00:04,2
1,Company representative Dave Campbell is talkin...,00:06 - 00:18,-1
2,A GMC employee is removing a tire of a car,00:21 - 00:24,2
3,An example is provided of tires with a good an...,00:32 - 00:37,1
4,GMC recommends rotating tires every 7500 miles,00:38 - 00:43,2


In [85]:
def match_descriptions(df1, df2, threshold: float = 0.5):
    model = SentenceTransformer("all-MiniLM-L6-v2")

    #Encode all descriptions
    emb1 = model.encode(df1["description"].tolist(), convert_to_tensor=True)
    emb2 = model.encode(df2["description"].tolist(), convert_to_tensor=True)

    #Cosine similarity matrix
    sim_matrix = util.cos_sim(emb1, emb2)

    matches = []
    
    for index1 in range(len(df1)):
        best_index2  = torch.argmax(sim_matrix[index1]).item()
        best_score = sim_matrix[index1][best_index2].item()

        if best_score >= threshold:
            matches.append({
                "description_1": df1.iloc[index1]["description"],
                "description_2": df2.iloc[best_index2]["description"],
                "similarity": round(best_score, 3),
                "score_1": df1.iloc[index1]["score"],
                "score_2": df2.iloc[best_index2]["score"],
                "timestamp_1": df1.iloc[index1]["timestamp"],
                "timestamp_2": df2.iloc[best_index2]["timestamp"],
            })
            sim_matrix[:, best_index2] = -1  #Put to -1 so it can't be matched again

    return pd.DataFrame(matches)

matched = match_descriptions(annotator_1, annotator_2)

In [86]:
print(matched)

                                       description_1  \
0  Branding screen of GMC's tire replacement service   
1               Person talking in front of a GMC car   
2                   Technician working on a car tire   
3    Comparison of a tire with a good and bad thread   
4  Displayed text recommending tire rotation ever...   
5  Demonstration of checking tire tread depth usi...   
6                        Tire spinning while mounted   
7                      People interacting near a car   
8  Branding screen of GMC's tire replacement service   

                                       description_2  similarity  score_1  \
0  Advertising title screen for the company "GMC"...       0.775        0   
1  Company representative Dave Campbell is talkin...       0.658        0   
2         A GMC employee is removing a tire of a car       0.602        1   
3  An example is provided of tires with a good an...       0.625        2   
4     GMC recommends rotating tires every 7500 miles  

In [95]:
#Consistency in selected events with Cohen's Kappa

n_matched = len(matched)
#Number of events selected by annotator 1/2 only
n_only_a1 = len(annotator_1) - n_matched
n_only_a2 = len(annotator_2) - n_matched

#Makes two binary vector where each position indicates if one event is selected or not
a1 = [1] * n_matched + [1] * n_only_a1 + [0] * n_only_a2
a2 = [1] * n_matched + [0] * n_only_a1 + [1] * n_only_a2

kappa = cohen_kappa_score(a1, a2)
print(f"Cohen's Kappa on event selection: {kappa:.3f}")

Cohen's Kappa on event selection: 0.000


In [ ]:
#Temporal boundaries

#Converts "00:44 - 00:58" to (44, 58) in seconds (so, 01:05 is 65)
def parse_timestamp(ts):
    start, end = ts.split(" - ")
    def to_seconds(t):
        m, s = t.strip().split(":")
        return int(m) * 60 + int(s)
    return to_seconds(start), to_seconds(end)

#Computes IoU
def iou(ts1,ts2):
    s1, e1 = parse_timestamp(ts1)
    s2, e2 = parse_timestamp(ts2)
    
    overlap = max(0, min(e1, e2) - max(s1, s2))
    union = max(e1, e2) - min(s1, s2)
    
    return overlap / union if union > 0 else 0.0

#Apply to matched pairs
matched["iou"] = matched.apply(
    lambda row: iou(row["timestamp_1"], row["timestamp_2"]), axis=1
)

print(f"Mean IoU on temporal boundaries: {matched['iou'].mean():.3f}")

Mean IoU on temporal boundaries: 0.790


In [ ]:
#Quadratic penalizes larger disagreements
kappa_scores = cohen_kappa_score(matched["score_1"], matched["score_2"], weights="quadratic") 
print(f"Cohen's Kappa on subjectivity ratings: {kappa_scores:.3f}")

Kappa on saliency scores: 0.407


In [ ]:
#Missed events by annotator 1

matched_a2 = set(matched["description_2"])
missed = annotator_2[~annotator_2["description"].isin(matched_a2)]

print(missed["description"].tolist())

['Demonstrated how to do this with a penny', 'Sneaky potholes and curbs are shown', '"Certified Service" on a building is shown', 'While Dave Campbell is talking gmccertifiedservice.com pops up']


In [ ]:
#Missed events by annotator 2

matched_r1 = set(matched["description_1"])
missed = annotator_1[~annotator_1["description"].isin(matched_r1)]

print(missed["description"].tolist())

[]
